# OVITO viewer for the adhesion workflow

Shows one stage's trajectory (`runs/09_pull` or `runs/05_compress`) with OVITO in the browser.
Run it from `workflows/adhesion` with a Python that has `ovito`, `ipywidgets` and `mdtraj`
(`pip install ovito jupyterlab ipywidgets`), started as `jupyter lab`.

* Topology (types, charges, bonds): `final.data` of that stage.
* Positions per frame: `trajectory.dcd`, converted once by `dcd_to_ovito.py` to
  `trajectory_ovito.dump` (`id x y z`, cell 0..L).
* Add or change modifiers in cell 3, then re-run cells 3 and 4.
* Cell 4: rotate/zoom with the mouse, reset the camera with the axis buttons
  ((1 0 0), (0 1 0), (0 0 1): orthographic; initial: the starting perspective), pick a frame with the slider, tick *wrap* to fold
  atoms back into the periodic cell, `save PNG` for the frame shown, and `export MP4` for a
  frame range from the current camera (files go to `RUN_DIR`). Needs ipykernel < 7
  (`pip install "ipykernel<7"`): newer kernels handle widget events in another thread, where
  OVITO crashes.
* Also in cell 4: the current camera as a `Viewport(...)` call (copy it into a script), a
  renderer choice for PNG/MP4 (OpenGL, or Tachyon ray tracing with optional ambient
  occlusion: slower, CPU), and for `09_pull` the pull distance, spring and film positions
  and the force at the frame shown, next to a force curve. Needs matplotlib.


In [ ]:
# 1. Which stage to show (relative to workflows/adhesion, or an absolute path)
RUN_DIR = 'projects/pmma_squeeze/runs/09_pull'
import faulthandler
faulthandler.enable(open('/home/suehara/ovito_crash.log', 'w'), all_threads=True)

In [ ]:
# 2. DCD -> OVITO dump (skipped when the dump is newer than the DCD)
from pathlib import Path
from dcd_to_ovito import convert

d = Path(RUN_DIR)
dump = convert(d)
print(f'using {dump}')


In [ ]:
# 3. The OVITO pipeline: data file + trajectory + modifiers
import numpy as np
from ovito.io import import_file
from ovito.modifiers import LoadTrajectoryModifier, SliceModifier, WrapPeriodicImagesModifier

pipeline = import_file(str(d / 'final.data'), atom_style='full')
traj = LoadTrajectoryModifier()
traj.source.load(str(dump))
pipeline.modifiers.append(traj)

# Wrap atoms back into the periodic cell (off at first; the checkbox in cell 4 switches it).
# Bonds that then cross the cell boundary are drawn cut at the boundary.
wrap = WrapPeriodicImagesModifier()
wrap.enabled = False
pipeline.modifiers.append(wrap)

# Types are force-field names (sc4, osi, c3, ...): colour and size them by element, from the mass.
# Silica types (IFF: sc4, oc23, oc24, hoy) are drawn a little paler than the polymer.
SILICA = {'sc4', 'oc23', 'oc24', 'hoy'}
ELEMENT = [(1.2, 'H', (0.95, 0.95, 0.95), 0.30), (13.0, 'C', (0.35, 0.35, 0.35), 0.55),
           (14.5, 'N', (0.20, 0.30, 0.90), 0.55), (17.0, 'O', (0.90, 0.15, 0.10), 0.55),
           (29.0, 'Si', (0.95, 0.75, 0.25), 0.75)]
masses = {}
for line in (d / 'final.data').read_text().split('Masses', 1)[1].strip().splitlines():
    w = line.split()
    if not w or not w[0].isdigit():
        break
    masses[int(w[0])] = float(w[1])

def style_types(frame, data):
    types = data.particles_.particle_types_
    for t in types.types:
        m = masses.get(t.id, 12.0)
        for limit, el, color, radius in ELEMENT:
            if m < limit:
                break
        c = np.array(color)
        if t.name in SILICA:
            c = 0.55 * c + 0.45
        tt = types.make_mutable(t)
        tt.color, tt.radius = tuple(c), radius
pipeline.modifiers.append(style_types)

# Pulled atoms (09 only) in blue: set MARK_PULLED = True to colour them.
MARK_PULLED = False
pulled_file = d / 'pulled_atoms.txt'
if MARK_PULLED and pulled_file.exists():
    pulled = np.loadtxt(pulled_file, dtype=int, ndmin=1)            # 0-based
    def mark_pulled(frame, data):
        col = data.particles_.create_property('Color')
        base = np.array([t.color for t in data.particles.particle_types.types])
        idx = {t.id: k for k, t in enumerate(data.particles.particle_types.types)}
        ty = np.asarray(data.particles.particle_types)
        col[...] = base[[idx[x] for x in ty]]
        col[pulled] = (0.25, 0.45, 1.0)
    pipeline.modifiers.append(mark_pulled)

# Example: a 20-A thick slab through the middle of the cell, to see inside the film.
# pipeline.modifiers.append(SliceModifier(normal=(0, 1, 0), distance=20.0, slab_width=20.0))

print('frames:', pipeline.num_frames)


In [ ]:
# 4. Interactive viewer (drag to rotate, wheel to zoom), axis views, frame slider, wrap,
#    renderer choice, PNG and MP4, and (for 09_pull) the pull position and force beside it.
# The scene holds a display-only pipeline whose data is the chosen frame, computed
# explicitly by pipeline.compute(frame). The viewer widget does not follow the
# animation frame by itself, so each change puts that frame's data in the scene
# and sends it with refresh(); the camera stays where you left it.
#
# OVITO (Qt) must only be called from the kernel's main thread. ipykernel >= 7
# handles widget messages in another thread (a "subshell"), and OVITO called from
# there crashes the kernel, so every button and slider hands its work to the main
# thread's event loop. Rotating the view is handled inside OVITO's own widget and
# is only safe with ipykernel < 7 (see the check below).
import asyncio
import csv
import io
import json
import threading
import time
import ipykernel
import ipywidgets as w
import numpy as np
from IPython.display import display
import ovito
from ovito.pipeline import Pipeline, StaticSource
from ovito.vis import Viewport, OpenGLRenderer, TachyonRenderer

if int(ipykernel.__version__.split('.')[0]) >= 7:
    print(f'NOTE: ipykernel {ipykernel.__version__}: rotating the view can crash the kernel. '
          'Install ipykernel<7 in this environment (pip install "ipykernel<7") and restart the kernel.')
main_loop = asyncio.get_event_loop()
main_thread = threading.main_thread()

def on_main(fn, *args):
    if threading.current_thread() is main_thread:
        fn(*args)
    else:
        main_loop.call_soon_threadsafe(fn, *args)

nframes = pipeline.num_frames
for p in list(ovito.scene.pipelines):
    p.remove_from_scene()
shown = Pipeline(source=StaticSource(data=pipeline.compute(0)))
shown.add_to_scene()

vp = Viewport(type=Viewport.Type.Perspective, camera_dir=(0.0, 1.0, 0.0))
vp.zoom_all()
view = ovito.gui.create_ipywidget(vp, layout=w.Layout(width='100%', height='650px'))
slider = w.IntSlider(min=0, max=nframes - 1, value=0, description='frame',
                     continuous_update=False, layout=w.Layout(width='100%'))
play = w.Play(min=0, max=nframes - 1, interval=1500, description='play')
w.jslink((play, 'value'), (slider, 'value'))
wrap_box = w.Checkbox(value=wrap.enabled, description='wrap into the cell (x, y, z periodic)', indent=False)
status = w.Label(f'frame 0 / {nframes - 1}')

# --- the current camera, as the Viewport(...) call that reproduces it ---
# The browser sends the camera back whenever the view is rotated or zoomed; its
# matrix is OVITO's camera_tm (3 x 4): the camera looks along -column 2 and sits at column 3.
camera_text = w.Text(value='', description='camera', disabled=False,
                     layout=w.Layout(width='100%'), style={'description_width': '60px'})

def show_camera(_=None):
    c = view._camera_params
    if not c:
        return
    m = np.array(c['matrix'])
    direction = -m[:, 2]
    pos = m[:, 3]
    kind = 'Perspective' if c['perspective'] else 'Ortho'
    camera_text.value = (f'Viewport(type=Viewport.Type.{kind}, '
                         f'camera_dir=({direction[0]:.3f}, {direction[1]:.3f}, {direction[2]:.3f}), '
                         f'camera_pos=({pos[0]:.2f}, {pos[1]:.2f}, {pos[2]:.2f}), fov={c["fov"]:.4f})')
view.observe(show_camera, names='_camera_params')
show_camera()

# --- camera presets: along a cell axis (orthographic), or back to the initial perspective ---
VIEWS = {  # label: (camera direction, up, orthographic)
    '(1 0 0)': ((-1, 0, 0), (0, 0, 1), True),     # looking along -x from the +x side, z up
    '(0 1 0)': ((0, -1, 0), (0, 0, 1), True),     # looking along -y from the +y side, z up
    '(0 0 1)': ((0, 0, -1), (0, 1, 0), True),     # looking down from +z, y up
    'initial': ((0.3, 1.0, -0.25), (0, 0, 1), False),
}

def set_view(label):
    direction, up, ortho = VIEWS[label]
    vp.type = Viewport.Type.Ortho if ortho else Viewport.Type.Perspective
    vp.camera_dir = direction
    vp.camera_up = up
    vp.zoom_all()
    view.send_camera()              # push the new camera to the browser (and to the camera box)
    view.refresh()

view_buttons = []
for label in VIEWS:
    b = w.Button(description=label, layout=w.Layout(width='90px'),
                 tooltip='orthographic view along this axis' if VIEWS[label][2] else 'the starting perspective view')
    b.on_click(lambda _, label=label: on_main(set_view, label))
    view_buttons.append(b)

# --- pull: position and force at the frame shown (09_pull only) ---
# Frame k of the trajectory is step k * (steps / (frames - 1)): the pull stage writes
# the first frame at step 0 and then one every dcd_every steps.
pull_csv = d / 'pull_force.csv'
pull_panel = None
if pull_csv.exists():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    with open(pull_csv) as fh:
        rows = np.array([[float(r['step']), float(r['time_ps']), float(r['z_ref_A']), float(r['z_com_A']),
                          float(r['extension_A']), float(r['force_nN'])] for r in csv.DictReader(fh)])
    p_step, p_time, p_zref, p_zcom, p_ext, p_force = rows.T
    p_dist = p_zref - p_zref[0]                              # how far the spring's reference has moved
    total_steps = json.loads((d / 'summary.json').read_text()).get('steps', p_step[-1])
    # The instantaneous force is noisy: a running mean over about 1 ps (the dt is in the csv).
    dt_ps = (p_time[-1] - p_time[0]) / max(1, len(p_time) - 1)
    win = max(1, int(round(1.0 / dt_ps))) if dt_ps > 0 else 1
    kernel = np.ones(win)
    p_smooth = (np.convolve(p_force, kernel, mode='same')
                / np.convolve(np.ones_like(p_force), kernel, mode='same'))   # ends: mean of what is there
    keep = slice(None, None, max(1, len(p_dist) // 20000))  # at most ~20k points to draw

    pull_plot = w.Image(format='png', layout=w.Layout(width='100%'))
    pull_info = w.HTML()

    def pull_row(frame):
        step = frame * total_steps / max(1, nframes - 1)
        return int(np.clip(np.searchsorted(p_step, step), 0, len(p_step) - 1))

    def draw_pull(frame):
        i = pull_row(frame)
        fig, ax = plt.subplots(figsize=(5.0, 3.6), dpi=110)
        ax.plot(p_dist[keep], p_force[keep], color='0.65', lw=0.5, label='force')
        ax.plot(p_dist[keep], p_smooth[keep], color='C0', lw=1.4, label=f'mean over {win * dt_ps:.2g} ps')
        ax.axvline(p_dist[i], color='C3', lw=1)
        ax.plot([p_dist[i]], [p_smooth[i]], 'o', color='C3')
        ax.axhline(0, color='k', lw=0.5)
        ax.set_xlabel('pull distance (z_ref − z_ref,0), Å')
        ax.set_ylabel('force on the film, nN')
        ax.legend(loc='upper right', fontsize=8, frameon=False)
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png')
        plt.close(fig)
        pull_plot.value = buf.getvalue()
        pull_info.value = (
            f'<table style="font-family:monospace">'
            f'<tr><td>frame</td><td>{frame} / {nframes - 1}</td></tr>'
            f'<tr><td>step</td><td>{int(p_step[i]):,} ({p_time[i]:.1f} ps)</td></tr>'
            f'<tr><td>pull distance</td><td>{p_dist[i]:.2f} Å</td></tr>'
            f'<tr><td>z_ref (spring)</td><td>{p_zref[i]:.2f} Å</td></tr>'
            f'<tr><td>z_com (pulled atoms)</td><td>{p_zcom[i]:.2f} Å</td></tr>'
            f'<tr><td>extension</td><td>{p_ext[i]:.2f} Å</td></tr>'
            f'<tr><td>force</td><td>{p_force[i]:.2f} nN</td></tr>'
            f'<tr><td>force, mean</td><td>{p_smooth[i]:.2f} nN</td></tr></table>')
    draw_pull(0)
    pull_panel = w.VBox([pull_plot, pull_info], layout=w.Layout(flex='2 1 0%', min_width='320px'))

def draw_frame(f):
    if f != slider.value:           # the slider has moved on; draw only the latest frame
        return
    shown.source.data = pipeline.compute(f)
    view.refresh()
    if pull_panel is not None:
        draw_pull(f)
    status.value = f'frame {f} / {nframes - 1}' + (' (wrapped)' if wrap.enabled else '')

def show_frame(change):
    status.value = f'frame {change["new"]} / {nframes - 1}: computing...'
    on_main(draw_frame, change['new'])
slider.observe(show_frame, names='value')

def set_wrap(on):
    wrap.enabled = on
    draw_frame(slider.value)
wrap_box.observe(lambda ch: on_main(set_wrap, ch['new']), names='value')

# --- renderer for PNG and MP4: OpenGL (fast) or Tachyon (ray traced, CPU, slower) ---
renderer_pick = w.Dropdown(options=['OpenGL', 'Tachyon', 'Tachyon + ambient occlusion'], value='OpenGL',
                           description='renderer', layout=w.Layout(width='330px'))

def make_renderer():
    if renderer_pick.value == 'OpenGL':
        return OpenGLRenderer()
    return TachyonRenderer(antialiasing=True, ambient_occlusion=renderer_pick.value.endswith('occlusion'))

def tag():
    return ('_wrap' if wrap.enabled else '') + ('' if renderer_pick.value == 'OpenGL' else '_tachyon')

# --- still image: the frame shown, from the current camera ---
size_w = w.BoundedIntText(value=1600, min=160, max=7680, step=16, description='width', layout=w.Layout(width='160px'))
size_h = w.BoundedIntText(value=1000, min=120, max=4320, step=16, description='height', layout=w.Layout(width='160px'))
save = w.Button(description='save PNG', tooltip='the frame shown, from the current camera')

def render_png():
    out = d / f'frame_{slider.value:04d}{tag()}.png'
    status.value = f'rendering {out.name} ({renderer_pick.value})...'
    t0 = time.time()
    try:
        vp.render_image(size=(size_w.value, size_h.value), filename=str(out), background=(1, 1, 1),
                        renderer=make_renderer())
    except RuntimeError as e:      # e.g. no working Vulkan driver: the viewer itself still works
        status.value = f'PNG not saved: {str(e).splitlines()[0]}'
        return
    status.value = f'saved {out} ({time.time() - t0:.0f} s)'
save.on_click(lambda _: on_main(render_png))

# --- movie: the chosen frame range from the current camera, wrap as set ---
movie_range = w.IntRangeSlider(value=(0, nframes - 1), min=0, max=nframes - 1, description='movie frames',
                               continuous_update=False, layout=w.Layout(width='100%'))
every = w.BoundedIntText(value=1, min=1, max=max(1, nframes), description='every nth', layout=w.Layout(width='160px'))
fps = w.BoundedIntText(value=15, min=1, max=120, description='fps', layout=w.Layout(width='160px'))
movie = w.Button(description='export MP4', tooltip='render the frame range from the current camera')

def render_movie():
    a, b = movie_range.value
    n = len(range(a, b + 1, every.value))
    out = d / f'movie_{a:04d}-{b:04d}{tag()}.mp4'
    status.value = f'rendering {n} frames to {out.name} ({renderer_pick.value}; the kernel is busy until it is done)...'
    movie.disabled = True
    # render_anim evaluates the scene's pipelines frame by frame, so the trajectory
    # pipeline stands in for the one-frame display pipeline while the movie is made.
    shown.remove_from_scene()
    pipeline.add_to_scene()
    t0 = time.time()
    try:
        vp.render_anim(filename=str(out), size=(size_w.value, size_h.value), fps=fps.value,
                       range=(a, b), every_nth=every.value, background=(1, 1, 1), renderer=make_renderer())
        status.value = f'saved {out} ({n} frames, {time.time() - t0:.0f} s)'
    except RuntimeError as e:
        status.value = f'movie not saved: {str(e).splitlines()[0]}'
    finally:
        pipeline.remove_from_scene()
        shown.add_to_scene()
        movie.disabled = False
movie.on_click(lambda _: on_main(render_movie))

viewer = w.VBox([view], layout=w.Layout(flex='3 1 0%'))
top = w.HBox([viewer, pull_panel]) if pull_panel is not None else viewer
display(w.VBox([top, camera_text, w.HBox([play, slider]), w.HBox([w.Label('view'), *view_buttons]), wrap_box,
                w.HBox([renderer_pick, size_w, size_h, save]),
                movie_range, w.HBox([every, fps, movie]), status]))
